# Build It: Retrieval-Augmented Generation (RAG) from Scratch

Build a complete 6-step RAG pipeline using the Python standard library without external frameworks.

```
[Documents] -> [1. Chunking] -> [2. TF-IDF Embedding] -> [Indexed Vectors]
                                                                |
[User Query] -> [Query Vector] -> [3. Vector Search] <----------+
                                          |
                                  [Top-K Chunks]
                                          |
                                  [4. Build Prompt]
                                          |
                               [5. Complete Pipeline]
                                          |
                                  [6. Generation] -> [Final Answer]
```


In [ ]:
import math
from collections import Counter
from typing import List, Tuple, Dict, Any


---
## Step 1: Document Chunking

Divide raw text into segments of `chunk_size` words with an `overlap` window to preserve sentence context across boundaries.

### Task:
Implement `chunk_text(text, chunk_size=200, overlap=50)`.
- Input: `text` (str), `chunk_size` (int), `overlap` (int).
- Output: `List[str]`.
- Step size: increment starting index by `chunk_size - overlap` each iteration.


In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Split text into chunks of `chunk_size` words with `overlap` words between consecutive chunks.

    Example:
        >>> chunk_text("A B C D E F", chunk_size=4, overlap=2)
        ['A B C D', 'C D E F']
    """
    words = text.split()
    chunks = []
    step = max(1, chunk_size - overlap)
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start : start + chunk_size]))
    return chunks


In [ ]:
# Test Step 1: Chunking
sample_text = " ".join([f"word_{i}" for i in range(100)])
chunks = chunk_text(sample_text, chunk_size=30, overlap=10)

assert len(chunks) == 5, f"Expected 5 chunks, got {len(chunks)}"
assert len(chunks[0].split()) == 30, "Chunk 0 must contain 30 words"
assert chunks[0].split()[-10:] == chunks[1].split()[:10], "Overlap between chunk 0 and 1 must match"

print("Step 1 passed.")


---
## Step 2: TF-IDF Embeddings

Convert text into numerical vectors using Term Frequency-Inverse Document Frequency:
- Term Frequency (TF): `count(word in doc) / total_words(doc)`
- Inverse Document Frequency (IDF): `ln((N + 1) / (DF(word) + 1)) + 1`
- TF-IDF Vector: element-wise product `TF * IDF` over vocabulary.

### Task:
Implement `compute_tf`, `compute_idf`, and `tfidf_embed`.


In [ ]:
def compute_tf(text: str, vocab: List[str]) -> List[float]:
    """
    Compute Term Frequency vector for text against vocabulary.
    TF(t, d) = count(t in d) / total_words(d)

    Example:
        >>> compute_tf("cat dog cat", ["cat", "dog", "fish"])
        [0.6666666666666666, 0.3333333333333333, 0.0]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Normalize text to lowercase and split into words
    # 2. Handle empty text edge case (return zero vector matching vocabulary length)
    # 3. Count frequencies of words in the text
    # 4. Calculate relative frequency for each word in vocabulary (occurrences / total words)
    pass
    ### END CODE ###


def compute_idf(documents: List[str], vocab: List[str]) -> List[float]:
    """
    Compute Inverse Document Frequency vector across documents.
    IDF(t) = ln((N + 1) / (DF(t) + 1)) + 1

    Example:
        >>> compute_idf(["cat dog", "cat fish"], ["cat", "dog", "fish"])
        [1.0, 1.4054651081081644, 1.4054651081081644]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Determine total document count N
    # 2. For each word in vocabulary, count how many documents contain it
    # 3. Apply smoothed IDF formula: log((N + 1) / (doc_count + 1)) + 1.0
    # 4. Return list of IDF weights ordered by vocabulary
    pass
    ### END CODE ###


def tfidf_embed(text: str, vocab: List[str], idf: List[float]) -> List[float]:
    """
    Compute TF-IDF embedding vector as element-wise product of TF and IDF.

    Example:
        >>> tfidf_embed("cat dog", ["cat", "dog"], [1.0, 1.5])
        [0.5, 0.75]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Compute term frequency vector for input text
    # 2. Multiply each term frequency value by its corresponding IDF weight
    # 3. Return the resulting vector
    pass
    ### END CODE ###


In [ ]:
def compute_tf(text: str, vocab: List[str]) -> List[float]:
    """
    Compute Term Frequency vector for text against vocabulary.
    TF(t, d) = count(t in d) / total_words(d)

    Example:
        >>> compute_tf("cat dog cat", ["cat", "dog", "fish"])
        [0.6666666666666666, 0.3333333333333333, 0.0]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Lowercase and split text into words: words = text.lower().split()
    # 2. If len(words) == 0, return vector of zeros with length len(vocab)
    # 3. Count word occurrences using Counter
    # 4. Return list of [count.get(w, 0) / len(words) for w in vocab]
    pass
    ### END CODE ###


def compute_idf(documents: List[str], vocab: List[str]) -> List[float]:
    """
    Compute Inverse Document Frequency vector across documents.
    IDF(t) = ln((N + 1) / (DF(t) + 1)) + 1

    Example:
        >>> compute_idf(["cat dog", "cat fish"], ["cat", "dog", "fish"])
        [1.0, 1.4054651081081644, 1.4054651081081644]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. n = len(documents)
    # 2. For each word in vocab, count docs containing word:
    #      doc_count = sum(1 for d in documents if word in d.lower().split())
    # 3. Compute math.log((n + 1) / (doc_count + 1)) + 1.0
    pass
    ### END CODE ###


def tfidf_embed(text: str, vocab: List[str], idf: List[float]) -> List[float]:
    """
    Compute TF-IDF embedding vector as element-wise product of TF and IDF.

    Example:
        >>> tfidf_embed("cat dog", ["cat", "dog"], [1.0, 1.5])
        [0.5, 0.75]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # tf = compute_tf(text, vocab)
    # return [t * i for t, i in zip(tf, idf)]
    pass
    ### END CODE ###


In [ ]:
# Test Step 2: TF-IDF Embeddings
docs = [
    "order block indicates institutional buying",
    "institutional buying creates fair value gap",
    "risk management defines stop loss"
]
vocab = build_vocabulary(docs)

tf = compute_tf("institutional buying", vocab)
assert len(tf) == len(vocab), "TF vector length must match vocab length"
assert math.isclose(sum(tf), 1.0), "TF sum of 2-word text must equal 1.0"

idf = compute_idf(docs, vocab)
assert len(idf) == len(vocab), "IDF vector length must match vocab length"
assert idf[vocab.index("stop")] > idf[vocab.index("institutional")], "Rare words must have higher IDF"

emb = tfidf_embed("order block", vocab, idf)
assert len(emb) == len(vocab), "Embedding dimension must match vocab length"
assert any(v > 0 for v in emb), "Embedding of known words must have positive values"

print("Step 2 passed.")


---
## Step 3: Cosine Similarity and Vector Search

Cosine similarity measures the angle between two vectors:

$$\text{similarity}(A, B) = \frac{A \cdot B}{\Vert A \Vert \Vert B \Vert}$$

### Task:
Implement `search(query_embedding, stored_embeddings, top_k=5)`.


In [ ]:
# Pre-implemented helper: cosine similarity
def cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    Compute cosine similarity between two numerical vectors.

    Example:
        >>> cosine_similarity([1.0, 0.0], [1.0, 0.0])
        1.0
        >>> cosine_similarity([1.0, 0.0], [0.0, 1.0])
        0.0
    """
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)


In [ ]:
def search(query_embedding: List[float], stored_embeddings: List[List[float]], top_k: int = 5) -> List[Tuple[int, float]]:
    """
    Rank stored embeddings by cosine similarity to query_embedding.
    Returns top_k tuples of (index, similarity_score) sorted descending by score.

    Example:
        >>> search([1.0, 0.0], [[0.0, 1.0], [1.0, 0.0]], top_k=1)
        [(1, 1.0)]
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Compute cosine similarity between query embedding and each stored embedding
    # 2. Pair each calculated score with its original index
    # 3. Sort pairs in descending order of similarity score
    # 4. Return the top_k ranked pairs
    pass
    ### END CODE ###


In [ ]:
# Test Step 3: Vector Search
query_vec = [1.0, 0.0, 0.0]
stored_vecs = [
    [0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.7, 0.7, 0.0]
]
results = search(query_vec, stored_vecs, top_k=2)

assert len(results) == 2, "Must return top_k results"
assert results[0][0] == 1, "Top match must be index 1"
assert math.isclose(results[0][1], 1.0), "Similarity score for identical vector must be 1.0"
assert results[1][0] == 2, "Second match must be index 2"

print("Step 3 passed.")


---
## Step 4: Prompt Construction

Format retrieved chunks into a prompt with clear grounding constraints.

### Task:
Implement `build_rag_prompt(query, retrieved_chunks)`.


In [ ]:
def build_rag_prompt(query: str, retrieved_chunks: List[str]) -> str:
    """
    Format retrieved chunks into bounded context and construct full RAG prompt.

    Example:
        >>> prompt = build_rag_prompt("What is X?", ["Chunk A", "Chunk B"])
        >>> "[Source 1]\nChunk A" in prompt and "Question: What is X?" in prompt
        True
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Label each chunk with its 1-indexed source number (e.g. [Source 1], [Source 2])
    # 2. Join labeled chunks into a single context string separated by divider markers
    # 3. Construct and return prompt incorporating instructions, context, question, and answer prompt
    pass
    ### END CODE ###


In [ ]:
# Test Step 4: Prompt Construction
test_chunks = [
    "Order blocks indicate high institutional volume.",
    "Always set stop loss to protect trading capital."
]
test_query = "What is an order block?"
prompt_output = build_rag_prompt(test_query, test_chunks)

assert "[Source 1]" in prompt_output, "Prompt must include [Source 1]"
assert "[Source 2]" in prompt_output, "Prompt must include [Source 2]"
assert "Question: What is an order block?" in prompt_output, "Prompt must include query"
assert "Context:" in prompt_output and "Answer:" in prompt_output, "Prompt must include Context and Answer headers"

print("Step 4 passed.")


---
## Step 5: Complete RAG Pipeline

Encapsulate indexing and querying into a single `RAGPipeline` class:
- `index(documents)`: Chunks text, builds vocabulary, computes IDF, and stores embeddings.
- `query(question, top_k=5)`: Embeds question, performs vector search, and formats augmented prompt.


In [ ]:
class RAGPipeline:
    def __init__(self, chunk_size: int = 150, overlap: int = 30):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.chunks: List[str] = []
        self.embeddings: List[List[float]] = []
        self.vocab: List[str] = []
        self.idf: List[float] = []

    def index(self, documents: List[str]) -> None:
        """
        Index documents into vector store.

        Example:
            >>> p = RAGPipeline(chunk_size=10, overlap=2)
            >>> p.index(["Sample doc one", "Sample doc two"])
            >>> len(p.chunks) > 0
            True
        """
        ### YOUR CODE HERE ###
        # TODO:
        # 1. Chunk all input documents using self.chunk_size and self.overlap
        # 2. Store all generated chunks in self.chunks
        # 3. Build corpus vocabulary from all chunks and assign to self.vocab
        # 4. Compute IDF weights across all chunks and assign to self.idf
        # 5. Compute embedding for each chunk and store in self.embeddings
        pass
        ### END CODE ###

    def query(self, question: str, top_k: int = 3) -> Tuple[str, List[Tuple[str, float]]]:
        """
        Retrieve top_k chunks and build augmented prompt.

        Example:
            >>> prompt, retrieved = p.query("What is sample?", top_k=1)
            >>> isinstance(prompt, str) and len(retrieved) <= 1
            True
        """
        ### YOUR CODE HERE ###
        # TODO:
        # 1. Embed input question into query vector using self.vocab and self.idf
        # 2. Search stored embeddings for top_k most similar chunks
        # 3. Collect matching chunk texts along with their similarity scores
        # 4. Build augmented prompt using question and retrieved chunks
        # 5. Return tuple of (prompt, retrieved_list)
        pass
        ### END CODE ###


In [ ]:
# Test Step 5: Complete Pipeline
sample_corpus = [
    "Fair Value Gap is a three-candle price imbalance pattern.",
    "Risk management requires capping loss per trade at 1 to 2 percent."
]
pipeline = RAGPipeline(chunk_size=30, overlap=10)
pipeline.index(sample_corpus)

assert len(pipeline.chunks) > 0, "Pipeline must store indexed chunks"
assert len(pipeline.embeddings) == len(pipeline.chunks), "Embeddings count must match chunks count"

prompt, retrieved = pipeline.query("What is the risk limit?", top_k=1)
assert len(retrieved) == 1, "Must retrieve top_k chunks"
assert "Context:" in prompt, "Generated prompt must contain Context section"

print("Step 5 passed.")


---
## Step 6: Generation (Simulated)

Simulate model generation by selecting the sentence with highest query word overlap from retrieved context. Returns fallback message if no relevant sentence is found.

### Task:
Implement `simple_generate(prompt, retrieved_chunks)`.


In [ ]:
def simple_generate(prompt: str, retrieved_chunks: List[str]) -> str:
    """
    Extract sentence with highest query word overlap from retrieved context.
    Returns "I don't have enough information." if no overlap is found.

    Example:
        >>> simple_generate("Question: dog? Answer:", ["The dog ran. Cat slept."])
        'The dog ran'
        >>> simple_generate("Question: bird? Answer:", ["The dog ran."])
        "I don't have enough information."
    """
    ### YOUR CODE HERE ###
    # TODO:
    # 1. Extract query words from prompt segment following "question:"
    # 2. Iterate through all sentences across retrieved chunks
    # 3. Calculate word overlap count between question words and current sentence
    # 4. Track sentence with highest overlap score
    # 5. Return best sentence if overlap > 0, otherwise return fallback string
    pass
    ### END CODE ###


In [ ]:
# Test Step 6: Generation
chunks = [
    "Fair Value Gap occurs when candle wicks do not overlap.",
    "Risk management requires capping loss at 1 to 2 percent."
]

ans_1 = simple_generate("Question: What is the risk management limit? Answer:", chunks)
assert "1 to 2 percent" in ans_1, "Must extract matching sentence"

ans_2 = simple_generate("Question: What is the distance to Mars? Answer:", chunks)
assert ans_2 == "I don't have enough information.", "Must return fallback when out of context"

print("Step 6 passed.")


---
## End-to-End Demo


In [ ]:
knowledge_base = [
    "Order Block indicates high institutional trading volume before an aggressive move.",
    "Fair Value Gap represents market inefficiency formed across a 3-candle sequence.",
    "Revenge trading is an emotional response to recover losses, leading to severe drawdown."
]

rag = RAGPipeline(chunk_size=40, overlap=10)
rag.index(knowledge_base)

queries = [
    "What is an Order Block?",
    "What causes revenge trading?",
    "What is the capital of Japan?"
]

for q in queries:
    prompt, retrieved = rag.query(q, top_k=1)
    answer = simple_generate(prompt, [chunk for chunk, _ in retrieved])
    print(f"Query:  {q}")
    print(f"Answer: {answer}\n")
